# Experimento 2 — Fauna no objetivo y rechazo de sonidos

**Pregunta:** ¿los datos de otras especies y los paisajes sonoros permiten enseñar al modelo de las 16 aves de Hito 1 qué debe excluir?

Este experimento audita la cobertura y las etiquetas de BirdCLEF+ 2025; no entrena un modelo y no modifica el Hito 1. Distingue entre **otra fauna etiquetada** y **sonido ambiental**, y cruza `secondary_labels` con las 16 aves objetivo para cuantificar contaminación conocida. Una etiqueta primaria no garantiza ausencia de aves objetivo y los soundscapes disponibles no tienen etiquetas por especie/ventana, por lo que ninguno debe asumirse automáticamente como negativo.

Informe: [`docs/experimento_02_rechazo_fauna_ambiente.md`](../docs/experimento_02_rechazo_fauna_ambiente.md).

In [1]:
from pathlib import Path

import ast
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "src" / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "src" / "data" / "processed"
if not (RAW_DIR / "train.csv").is_file():
    raise FileNotFoundError(f"No se encontró el dataset BirdCLEF+ 2025 en {RAW_DIR}")

train = pd.read_csv(RAW_DIR / "train.csv")
taxonomy = pd.read_csv(RAW_DIR / "taxonomy.csv")
durations = pd.read_csv(PROCESSED_DIR / "train_durations.csv", usecols=["filename", "duration_s"])
data = train.merge(
    taxonomy[["primary_label", "scientific_name", "common_name", "class_name"]],
    on="primary_label",
    validate="m:1",
).merge(durations, on="filename", validate="1:1")
print(f"Entrenamiento: {len(data):,} grabaciones; {data.primary_label.nunique()} etiquetas")

Entrenamiento: 28,564 grabaciones; 206 etiquetas


## Cobertura disponible de otras clases taxonómicas

Se aplican a todos los taxones los filtros de duración mínima y rating XC bajo usados para construir el subconjunto de Hito 1. Se deduplica por URL para no contar dos veces el mismo recurso. El filtro no convierte una grabación en negativo validado: solo homogeneiza criterios básicos de calidad.

In [2]:
MIN_DURATION_S = 1.0
MIN_XC_RATING = 2.0
keep = (data.duration_s >= MIN_DURATION_S) & ~(
    (data.collection == "XC") & data.rating.between(0.1, MIN_XC_RATING - 0.01)
)
filtered = data.loc[keep].drop_duplicates("url", keep="first").copy()
non_birds = filtered.loc[filtered.class_name.isin(["Amphibia", "Insecta", "Mammalia"])].copy()
species_counts = non_birds.groupby(["class_name", "primary_label"]).size().rename("recordings")
class_summary = non_birds.groupby("class_name").agg(
    recordings=("filename", "size"),
    species=("primary_label", "nunique"),
    authors=("author", "nunique"),
    hours=("duration_s", lambda values: values.sum() / 3600),
)
class_summary["median_recordings_per_species"] = species_counts.groupby(level=0).median()
class_summary["species_below_20_recordings"] = species_counts.lt(20).groupby(level=0).sum()
class_summary["hours"] = class_summary.hours.round(2)
class_summary

,recordings,species,authors,hours,median_recordings_per_species,species_below_20_recordings
class_name,,,,,,
Amphibia,576,34,178,4.91,6.0,25
Insecta,154,17,19,4.90,5.0,14
Mammalia,174,9,93,1.65,5.0,7


In [3]:
selected = pd.read_csv(PROCESSED_DIR / "subset_recordings.csv")
selected_counts = selected.groupby("primary_label").size()
pooled_other_count = len(non_birds)
def has_secondary_labels(value):
    if pd.isna(value):
        return False
    return any(label for label in ast.literal_eval(value))

actual_secondary_count = int(non_birds.secondary_labels.map(has_secondary_labels).sum())

comparison = pd.DataFrame(
    {
        "measure": [
            "Selected Hito 1 bird recordings",
            "Number of selected bird classes",
            "Smallest selected bird class",
            "Largest selected bird class",
            "Other-fauna recordings after filters",
            "Other-fauna species",
            "Other-fauna species with <20 recordings",
            "Other-fauna recordings with secondary label(s)",
        ],
        "value": [
            len(selected),
            selected_counts.size,
            selected_counts.min(),
            selected_counts.max(),
            pooled_other_count,
            non_birds.primary_label.nunique(),
            int(species_counts.lt(20).sum()),
            int(actual_secondary_count),
        ],
    }
)
print(f"Desbalance 16 clases Hito 1: {selected_counts.max() / selected_counts.min():.2f}:1")
comparison

Desbalance 16 clases Hito 1: 6.77:1


,measure,value
0,Selected Hito 1 bird recordings,6759
1,Number of selected bird classes,16
2,Smallest selected bird class,145
3,Largest selected bird class,982
4,Other-fauna recordings after filters,904
5,Other-fauna species,60
6,Other-fauna species with <20 recordings,46
7,Other-fauna recordings with secondary label(s),5


## Contaminación conocida por las 16 aves objetivo

Se comparan las etiquetas secundarias disponibles en las grabaciones no aviares con las 16 etiquetas seleccionadas en Hito 1. Una coincidencia conocida se excluye de negativos monoetiqueta. La falta de coincidencia no certifica ausencia, porque las etiquetas secundarias tienen cobertura muy limitada.

In [4]:
target_codes = set(selected.primary_label.astype(str).unique())

def parse_label_codes(value):
    if pd.isna(value):
        return []
    if not value.strip():
        return []
    parsed = ast.literal_eval(value)
    if not isinstance(parsed, list):
        raise ValueError(f"Se esperaba una lista en secondary_labels, se recibió: {value!r}")
    return [str(label) for label in parsed if label]

non_birds["secondary_codes"] = non_birds.secondary_labels.map(parse_label_codes)
non_birds["target_secondary_codes"] = non_birds.secondary_codes.map(
    lambda labels: sorted(target_codes.intersection(labels))
)
target_common_names = taxonomy.set_index("primary_label").common_name.to_dict()
known_contaminated = non_birds.loc[non_birds.target_secondary_codes.map(bool)].copy()
known_contaminated["target_secondary_names"] = known_contaminated.target_secondary_codes.map(
    lambda labels: [target_common_names[label] for label in labels]
)
contamination_summary = pd.DataFrame(
    {
        "measure": [
            "Non-bird recordings after filters",
            "Recordings with any secondary label",
            "Recordings with a secondary label matching a target bird",
            "Recordings without a recorded target-bird secondary label (not verified negatives)",
        ],
        "value": [
            len(non_birds),
            int(non_birds.secondary_codes.map(bool).sum()),
            len(known_contaminated),
            len(non_birds) - len(known_contaminated),
        ],
    }
)
display(contamination_summary)

if known_contaminated.empty:
    print("No target-bird secondary labels were recorded in this filtered subset.")
else:
    display(
        known_contaminated[
            ["filename", "primary_label", "target_secondary_codes", "target_secondary_names", "author", "collection"]
        ]
    )

,measure,value
0,Non-bird recordings after filters,904
1,Recordings with any secondary label,5
2,Recordings with a secondary label matching a t...,1
3,Recordings without a recorded target-bird seco...,903


,filename,primary_label,target_secondary_codes,target_secondary_names,author,collection
185,22973/XC892932.ogg,22973,[compau],[Common Pauraque],Chris Harrison,XC


## Soundscapes: volumen no equivale a negativos etiquetados

La tabla de metadatos de soundscapes describe características técnicas y de adquisición. Se comprueba su esquema para no inferir ausencia de especies a partir de que no haya una etiqueta en ese archivo.

In [5]:
soundscapes = pd.read_csv(PROCESSED_DIR / "train_soundscapes_meta.csv")
label_like_columns = [
    column for column in soundscapes.columns
    if any(token in column.lower() for token in ("label", "species", "annotation", "event"))
]
candidate_windows = int((soundscapes.duration_s // 5).sum())
soundscape_summary = pd.Series(
    {
        "archivos": len(soundscapes),
        "horas": round(soundscapes.duration_s.sum() / 3600, 1),
        "sitios": soundscapes.site.nunique(),
        "duración_mínima_s": soundscapes.duration_s.min(),
        "duración_máxima_s": soundscapes.duration_s.max(),
        "ventanas_candidatas_de_5_s": candidate_windows,
        "columnas_de_etiquetado_en_metadatos": label_like_columns,
    },
    name="soundscapes",
)
soundscape_summary

archivos                                 9726
horas                                   162.1
sitios                                     56
duración_mínima_s                        60.0
duración_máxima_s                        60.0
ventanas_candidatas_de_5_s             116712
columnas_de_etiquetado_en_metadatos        []
Name: soundscapes, dtype: object

## Propuesta de curación y uso de clases de rechazo

### Selección de `otra_fauna`

Para cada grabación no aviar, revisar el audio original y asignar `other_fauna_only`, `target_bird_present`, `mixed_target_and_other`, `uncertain` o `not_other_fauna_confirmed`. Solo `other_fauna_only` puede ser negativo monoetiqueta validado; los eventos mixtos requieren multilabel o exclusión del negativo. La etiqueta primaria no basta. Mantener el taxón original como subtipo y agruparlo en `otra_fauna` solo para el objetivo de rechazo.

Los metadatos arrojan una coincidencia secundaria conocida con las 16 aves, pero las etiquetas secundarias están incompletas: este conteo es un mínimo de contaminación registrada, no una estimación de prevalencia. Se recomienda revisar los 904 archivos filtrados (11,5 horas de escucha como mínimo), con doble anotación de validación/test y adjudicación de desacuerdos. Si no es posible revisarlos todos, mantener los no revisados como `uncertain`, nunca como negativos limpios.

### Sonidos ambientales de soundscapes

Usar los audios de El Silencio para anotar clases ambientales independientes: `rain` (lluvia), `water` (corrientes/fuentes de agua), `human_voice` (voces), `anthropogenic_other` (motores/herramientas), `wind`, además de `insect`, `amphibian`, `mammal` y `unknown_environment`. Seleccionar una muestra piloto estratificada por sitio, fecha y hora; anotar inicio/fin, tipo, confianza y presencia de aves objetivo. No inferir el tipo de sonido ni ausencia de aves a partir de metadatos.

Solo ventanas ambientales revisadas y sin aves objetivo detectadas bajo un protocolo de escucha pueden servir como negativos de rechazo. Si coexiste un ave con lluvia/agua/voz, conservar etiquetas concurrentes o excluir de negativos puros. Dividir por archivo y retener sitios/fechas completos antes de recortar para evitar fuga; reportar falsas alarmas y rechazos por tipo ambiental.

- No convertir las 60 especies en 60 clases; la cobertura por especie es demasiado baja. Evaluar `otra_fauna` como clase agrupada solo después de curar contaminación.
- Mantener `otra_fauna` y `ambiente` como motivos de rechazo distintos; permitir coexistencia de etiquetas en audio mixto.
- Comparar baseline cerrado de 16 clases con umbral calibrado, clase `otra_fauna` curada y clases/subtipos ambientales verificados.

El término «otra fauna» no equivale a «silencio» ni a «ambiente sin aves». El modelo necesita una salida de rechazo explícita —umbral calibrado o clases de no objetivo con etiquetas verificadas— porque un clasificador cerrado siempre predice una clase conocida.

### Referencias

1. [BirdCLEF+ 2025 — competencia y datos](https://www.kaggle.com/competitions/birdclef-2025/data).
2. Scheirer et al. (2013), [Toward Open Set Recognition](https://doi.org/10.1109/TPAMI.2012.256), *IEEE TPAMI*, 35(7), 1757–1772.
3. Datos locales: `train.csv`, `taxonomy.csv`, `train_durations.csv`, `subset_recordings.csv` y `train_soundscapes_meta.csv`.